In [1]:
# Cell 1: Student Configuration & Workspace Setup
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Anonuevo"
STUDENT_ID = "TUPM-26-1079"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

WORK_DIR = Path.cwd()
REPO_DIR = WORK_DIR.parent if WORK_DIR.name == "Lab7_NumPy_Pandas_Additional_Exercises" else WORK_DIR
DATA_DIR = REPO_DIR / "data"

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

assert measurement_path.is_file() and label_path.is_file(), "SECOM dataset files not found in data/."

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Random Seed:", RANDOM_SEED)

Student: Anonuevo TUPM-26-1079
Seed Digit: 9 | ID Sum: 25 | Random Seed: 261079


In [2]:
# Cell 2 - Exercise 1
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
total_rows = raw_measurements.shape[0]
total_cols = raw_measurements.shape[1]

# Make feature names Feature_000, Feature_001, etc.
feature_names = []
for i in range(total_cols):
    feature_names.append(f"Feature_{i:03d}")

# Find usable columns (not all missing and not constant)
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0

observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)

usable_indices = np.flatnonzero(~(all_missing | constant))

# Pick 5 + (SEED_DIGIT % 3) features
rng = np.random.default_rng(RANDOM_SEED)
num_to_pick = 5 + (SEED_DIGIT % 3)
chosen_idx = rng.choice(usable_indices, size=num_to_pick, replace=False)

chosen_data = raw_measurements[:, chosen_idx]
chosen_names = [feature_names[i] for i in chosen_idx]

# Calculate stats using NumPy
means = np.nanmean(chosen_data, axis=0)
medians = np.nanmedian(chosen_data, axis=0)
sds = np.nanstd(chosen_data, axis=0, ddof=1)
missing_pct = (np.isnan(chosen_data).sum(axis=0) / total_rows) * 100

# Calculate MAD and robust standardized values
abs_diff_from_median = np.abs(chosen_data - medians)
mad = np.nanmedian(abs_diff_from_median, axis=0)
safe_mad = np.where(mad > 0, mad, 1.0)
robust_z = np.where(mad > 0, (chosen_data - medians) / safe_mad, 0.0)

# Flag values > 3.5 (ignoring NaN)
not_missing = ~np.isnan(chosen_data)
flagged = not_missing & (np.abs(robust_z) > 3.5)
flagged_counts = flagged.sum(axis=0)
flagged_pct = (flagged_counts / not_missing.sum(axis=0)) * 100

# Verify using a simple loop
check_counts = []
for col in range(num_to_pick):
    count = 0
    for row in range(total_rows):
        if not np.isnan(chosen_data[row, col]) and abs(robust_z[row, col]) > 3.5:
            count = count + 1
    check_counts.append(count)

verified = list(flagged_counts) == check_counts

# Put results into a table and sort
quality_table = pd.DataFrame({
    "Feature": chosen_names,
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "MAD": mad,
    "Missing_Percentage": missing_pct,
    "Flagged_Count": flagged_counts,
    "Flagged_Percentage": flagged_pct
})
quality_table = quality_table.sort_values(by="Flagged_Percentage", ascending=False).reset_index(drop=True)

quality_table.to_csv(WORK_DIR / "measurement_quality.csv", index=False)

print("Selected Features:", ", ".join(chosen_names))
print("Highest Missing Percentage:", f"{quality_table['Missing_Percentage'].max():.2f}%")
print("Feature with Highest Flagged Percentage:", quality_table.iloc[0]["Feature"])
print("Total Flagged Observations:", int(flagged_counts.sum()))
print("Vectorized Results Verified:", verified)
print(quality_table.round(4))

Selected Features: Feature_318, Feature_105, Feature_018, Feature_517, Feature_107
Highest Missing Percentage: 64.96%
Feature with Highest Flagged Percentage: Feature_517
Total Flagged Observations: 372
Vectorized Results Verified: True
       Feature      Mean    Median  Sample_SD     MAD  Missing_Percentage  \
0  Feature_517    1.7389    1.1543     4.8907  0.3507             64.9649   
1  Feature_105   -0.0005   -0.0002     0.0032  0.0015              0.3829   
2  Feature_107   -0.0018    0.0000     0.0875  0.0490              0.3829   
3  Feature_318    3.1888    2.8989     1.2159  0.7504              0.0638   
4  Feature_018  190.0474  189.6642     2.7810  1.8763              0.1914   

   Flagged_Count  Flagged_Percentage  
0             85             15.4827  
1            115              7.3671  
2             93              5.9577  
3             54              3.4483  
4             25              1.5985  


In [3]:
# Cell 3 - Exercise 2
df = pd.DataFrame(raw_measurements, columns=feature_names)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

df.insert(0, "Example_ID", range(len(df)))
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

# Separate Pass and Fail rows
pass_rows = df[df["Outcome"] == "Pass"]
fail_rows = df[df["Outcome"] == "Fail"]

# Pick balanced sample size
rng = np.random.default_rng(RANDOM_SEED)
group_size = min(20 + ID_SUM, len(pass_rows), len(fail_rows))

pass_sample = pass_rows.sample(n=group_size, random_state=RANDOM_SEED)
fail_sample = fail_rows.sample(n=group_size, random_state=RANDOM_SEED)
balanced_df = pd.concat([pass_sample, fail_sample])

# Pick 5 usable features and fill missing values using full dataset medians
picked_idx = rng.choice(usable_indices, size=5, replace=False)
picked_features = [feature_names[i] for i in picked_idx]

full_medians = df[picked_features].median()
balanced_sample = balanced_df[["Example_ID"] + picked_features + ["Result", "Outcome"]].copy()
balanced_sample[picked_features] = balanced_sample[picked_features].fillna(full_medians)

# Group by Outcome
grouped = balanced_sample.groupby("Outcome")[picked_features].agg(["count", "mean", "median", "std"])

# Check if NumPy and Pandas give the same Pass means
np_means = balanced_sample[balanced_sample["Outcome"] == "Pass"][picked_features].to_numpy().mean(axis=0)
pd_means = grouped.loc["Pass", (slice(None), "mean")].to_numpy()
stats_match = np.allclose(np_means, pd_means)

# Calculate mean differences and pooled standardized mean difference
results_list = []
for feat in picked_features:
    n_pass = grouped.loc["Pass", (feat, "count")]
    n_fail = grouped.loc["Fail", (feat, "count")]
    mean_pass = grouped.loc["Pass", (feat, "mean")]
    mean_fail = grouped.loc["Fail", (feat, "mean")]
    med_pass = grouped.loc["Pass", (feat, "median")]
    med_fail = grouped.loc["Fail", (feat, "median")]
    sd_pass = grouped.loc["Pass", (feat, "std")]
    sd_fail = grouped.loc["Fail", (feat, "std")]

    abs_diff = abs(mean_fail - mean_pass)
    pooled_var = ((n_pass - 1) * (sd_pass ** 2) + (n_fail - 1) * (sd_fail ** 2)) / (n_pass + n_fail - 2)
    pooled_sd = np.sqrt(pooled_var)

    if pooled_sd > 0:
        abs_smd = abs_diff / pooled_sd
    else:
        abs_smd = 0.0

    results_list.append({
        "Feature": feat,
        "Pass_Count": int(n_pass),
        "Fail_Count": int(n_fail),
        "Pass_Mean": mean_pass,
        "Fail_Mean": mean_fail,
        "Pass_Median": med_pass,
        "Fail_Median": med_fail,
        "Pass_SD": sd_pass,
        "Fail_SD": sd_fail,
        "Abs_Mean_Diff": abs_diff,
        "Abs_SMD": abs_smd
    })

feature_comparison = pd.DataFrame(results_list)
feature_comparison = feature_comparison.sort_values(by="Abs_SMD", ascending=False).reset_index(drop=True)

balanced_sample.to_csv(WORK_DIR / "balanced_sample.csv", index=False)
feature_comparison.to_csv(WORK_DIR / "feature_comparison.csv", index=False)

print("Balanced Sample Size:", len(balanced_sample))
print("Selected Features:", ", ".join(picked_features))
print("Highest-Ranked Feature:", feature_comparison.iloc[0]["Feature"])
print("Absolute Standardized Mean Difference:", f"{feature_comparison.iloc[0]['Abs_SMD']:.4f}")
print("NumPy-Pandas Statistics Match:", stats_match)
print(feature_comparison[["Feature", "Pass_Mean", "Fail_Mean", "Abs_Mean_Diff", "Abs_SMD"]].round(4))

Balanced Sample Size: 90
Selected Features: Feature_318, Feature_105, Feature_018, Feature_517, Feature_107
Highest-Ranked Feature: Feature_318
Absolute Standardized Mean Difference: 0.2289
NumPy-Pandas Statistics Match: True
       Feature  Pass_Mean  Fail_Mean  Abs_Mean_Diff  Abs_SMD
0  Feature_318     2.9627     3.2148         0.2521   0.2289
1  Feature_517     3.8015     1.1790         2.6226   0.2221
2  Feature_107    -0.0254    -0.0346         0.0092   0.0996
3  Feature_105    -0.0002    -0.0005         0.0002   0.0843
4  Feature_018   190.0689   189.8712         0.1977   0.0707


In [4]:
# Cell 4 - Exercise 3
def run_analysis(seed):
    rng = np.random.default_rng(seed)
    pass_mask = labels == -1
    fail_mask = labels == 1

    # Find features with at least 10 non-missing values in Pass and Fail
    valid_list = []
    for idx in usable_indices:
        col = raw_measurements[:, idx]
        pass_valid = (~np.isnan(col[pass_mask])).sum()
        fail_valid = (~np.isnan(col[fail_mask])).sum()
        if pass_valid >= 10 and fail_valid >= 10:
            valid_list.append(idx)

    chosen = rng.choice(valid_list, size=5, replace=False)
    chosen_feats = [feature_names[i] for i in chosen]
    n_boot = 200 + 10 * SEED_DIGIT

    rows = []
    for feat in chosen_feats:
        col_raw = df[feat].to_numpy()
        col_filled = np.where(np.isnan(col_raw), np.nanmedian(col_raw), col_raw)

        # Strategy 1: Available-case (ignore NaN)
        p_avail = col_raw[pass_mask & ~np.isnan(col_raw)]
        f_avail = col_raw[fail_mask & ~np.isnan(col_raw)]
        diff_avail = f_avail.mean() - p_avail.mean()

        boot_diffs_1 = []
        for _ in range(n_boot):
            bp = rng.choice(p_avail, size=len(p_avail), replace=True)
            bf = rng.choice(f_avail, size=len(f_avail), replace=True)
            boot_diffs_1.append(bf.mean() - bp.mean())

        # Strategy 2: Full-dataset median imputation
        p_imp = col_filled[pass_mask]
        f_imp = col_filled[fail_mask]
        diff_imp = f_imp.mean() - p_imp.mean()

        boot_diffs_2 = []
        for _ in range(n_boot):
            bp = rng.choice(p_imp, size=len(p_imp), replace=True)
            bf = rng.choice(f_imp, size=len(f_imp), replace=True)
            boot_diffs_2.append(bf.mean() - bp.mean())

        rows.append({
            "Feature": feat,
            "Avail_Mean_Diff": diff_avail,
            "Avail_Boot_Mean": np.mean(boot_diffs_1),
            "Avail_CI_Low": np.percentile(boot_diffs_1, 2.5),
            "Avail_CI_High": np.percentile(boot_diffs_1, 97.5),
            "Imputed_Mean_Diff": diff_imp,
            "Imputed_Boot_Mean": np.mean(boot_diffs_2),
            "Imputed_CI_Low": np.percentile(boot_diffs_2, 2.5),
            "Imputed_CI_High": np.percentile(boot_diffs_2, 97.5),
            "Change_In_Diff": abs(diff_imp - diff_avail),
            "Sign_Changed": (diff_avail > 0) != (diff_imp > 0)
        })

    table = pd.DataFrame(rows)
    table["Rank_Avail"] = table["Avail_Mean_Diff"].abs().rank(ascending=False).astype(int)
    table["Rank_Imputed"] = table["Imputed_Mean_Diff"].abs().rank(ascending=False).astype(int)
    table["Rank_Changed"] = table["Rank_Avail"] != table["Rank_Imputed"]
    return table, chosen_feats, n_boot

robust_table, selected_ex3, boot_reps = run_analysis(RANDOM_SEED)
robust_table_check, _, _ = run_analysis(RANDOM_SEED)
is_reproducible = robust_table.equals(robust_table_check)

max_change = robust_table["Change_In_Diff"].max()
changed_signs = int(robust_table["Sign_Changed"].sum())
changed_ranks = int(robust_table["Rank_Changed"].sum())

robust_table.to_csv(WORK_DIR / "robustness_comparison.csv", index=False)

report_text = (
    "STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT\n"
    f"Student: {LAST_NAME} ({STUDENT_ID})\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Bootstrap Repetitions: {boot_reps}\n"
    f"Selected Features: {', '.join(selected_ex3)}\n"
    f"Largest Change in Mean Difference: {max_change:.4f}\n"
    f"Features with Changed Difference Sign: {changed_signs}\n"
    f"Features with Changed Ranking: {changed_ranks}\n"
    f"Reproducibility Verified: {is_reproducible}\n\n"
    "Interpretation:\n"
    "Filling missing values with medians changes group mean differences and rankings slightly compared to "
    "available-case analysis, showing why preprocessing choices should be checked before making engineering conclusions.\n"
)
(WORK_DIR / "sensitivity_report.txt").write_text(report_text, encoding="utf-8")

print("Selected Features:", ", ".join(selected_ex3))
print("Bootstrap Repetitions:", boot_reps)
print("Largest Change in Mean Difference:", f"{max_change:.4f}")
print("Features with Changed Difference Sign:", changed_signs)
print("Features with Changed Ranking:", changed_ranks)
print("Reproducibility Verified:", is_reproducible)
print(robust_table[["Feature", "Avail_Mean_Diff", "Imputed_Mean_Diff", "Change_In_Diff", "Sign_Changed", "Rank_Changed"]].round(4))

Selected Features: Feature_320, Feature_105, Feature_018, Feature_517, Feature_106
Bootstrap Repetitions: 290
Largest Change in Mean Difference: 0.2859
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Reproducibility Verified: True
       Feature  Avail_Mean_Diff  Imputed_Mean_Diff  Change_In_Diff  \
0  Feature_320           0.0004             0.0004          0.0000   
1  Feature_105           0.0000             0.0000          0.0000   
2  Feature_018          -0.0988            -0.0980          0.0008   
3  Feature_517          -0.4586            -0.1727          0.2859   
4  Feature_106          -0.0003            -0.0003          0.0000   

   Sign_Changed  Rank_Changed  
0         False         False  
1         False         False  
2         False         False  
3         False         False  
4         False         False  
